# Magnetostatics

### The Fields Inside a Dipole and a Quadrupole Magnet

`parallel_plates.py` in this session solved for the electrostatic potential
around a parallel plate capacitor. It filled a grid with zeros, pinned two
plates at fixed voltages, and then averaged each cell with its four neighbors
thousands of times until the numbers stopped changing. That averaging is a
solver for **Laplace's equation**, $\nabla^2 V = 0$.

This notebook makes the case that the *same solver, unchanged*, gives the
static magnetic field inside an accelerator magnet. Only the labels change:
voltage becomes a magnetic scalar potential, and copper plates become iron
pole faces.

We will build up to two magnets in particular:

1. A **dipole**, the magnet that bends a particle beam.
2. A **quadrupole**, the magnet that focuses it - the kind of magnet the
   Alternating Gradient Synchrotron (AGS) at Brookhaven was built around.

Along the way we derive *why* the pole faces of these magnets have the shapes
they do, which turns out to be the most useful single idea in magnet design.

---
## 1. Why magnetostatics is also a Laplace problem

Two of Maxwell's equations govern a static magnetic field:

$$\nabla \cdot \mathbf{B} = 0
\qquad\text{and}\qquad
\nabla \times \mathbf{B} = \mu_0 \mathbf{J}$$

The second one is the obstacle. A magnetic field generally has curl, so it is
not the gradient of anything, and there is no magnetic analogue of voltage.

But look at *where* we actually want the answer. The beam travels through the
bore of the magnet - the empty region between the pole faces. The coils and
the iron are outside that region. Inside the bore $\mathbf{J} = 0$, so there

$$\nabla \times \mathbf{B} = 0$$

A curl-free field **is** a gradient. So in the bore, and only in the bore, we
may write

$$\mathbf{B} = -\nabla \psi$$

where $\psi$ is the **magnetic scalar potential**. Substituting this into
$\nabla \cdot \mathbf{B} = 0$ gives

$$\boxed{\nabla^2 \psi = 0}$$

which is Laplace's equation, the identical PDE `parallel_plates.py` solved.

### A note on units

The textbook definition uses the $\mathbf{H}$ field,
$\mathbf{H} = -\nabla \phi_m$, which puts $\phi_m$ in amperes. Throughout this
notebook we instead use the **reduced** potential $\psi = \mu_0 \phi_m$, so
that $\mathbf{B} = -\nabla \psi$ directly and $\psi$ carries units of
tesla-meters. Every field then comes out in tesla with no bookkeeping.

### Why iron poles behave like capacitor plates

Iron has a relative permeability of several thousand. Inside it,
$\mathbf{H} = \mathbf{B}/\mu \approx 0$. The tangential component of
$\mathbf{H}$ is continuous across a boundary, so just outside the iron the
tangential $\mathbf{H}$ must also be nearly zero, which means:

> **$\mathbf{B}$ leaves an iron surface perpendicular to it, so an iron pole
> face is a surface of constant $\psi$.**

That is precisely the boundary condition a conductor imposes in
electrostatics. A pole face is an equipotential, exactly like a capacitor
plate, and the coils wrapped around it set its potential the way a battery
sets a plate's voltage.

In [ ]:
"""magnetostatics.ipynb"""

# Cell 01 - Imports and the electrostatic / magnetostatic dictionary

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.constants import mu_0
from scipy.ndimage import convolve, generate_binary_structure

# The five point Laplacian stencil: average the four nearest neighbors and
# ignore the center cell. This is the same kernel used in parallel_plates.py.
LAPLACE_KERNEL = generate_binary_structure(2, 1).astype(float) / 4
LAPLACE_KERNEL[1, 1] = 0.0

print(f"mu_0 = {mu_0:.6e} T m / A")
print(f"Relaxation kernel:\n{LAPLACE_KERNEL}")

pd.DataFrame(
    {
        "Electrostatics": [
            "voltage V",
            "E = -grad V",
            "div E = rho / eps_0",
            "Laplace: grad^2 V = 0",
            "conductor surface",
            "battery sets plate voltage",
        ],
        "Magnetostatics": [
            "scalar potential psi",
            "B = -grad psi",
            "div B = 0",
            "Laplace: grad^2 psi = 0",
            "iron pole face",
            "coil sets pole potential",
        ],
    },
    index=["potential", "field", "divergence", "PDE", "equipotential", "source"],
)

---
## 2. First, the other kind of dipole

The word "dipole" means two different things, and it is worth separating them
before going further.

To a physicist, a magnetic dipole is a **current loop shrunk to a point**, with
moment $\mathbf{m}$. This is the bar magnet of an introductory course. Its
scalar potential and field are

$$\psi(\mathbf{r}) = \frac{\mu_0}{4\pi}\frac{\mathbf{m}\cdot\hat{\mathbf{r}}}{r^2}
\qquad
\mathbf{B}(\mathbf{r}) = \frac{\mu_0}{4\pi}
\frac{3(\mathbf{m}\cdot\hat{\mathbf{r}})\hat{\mathbf{r}} - \mathbf{m}}{r^3}$$

Written out for $\mathbf{m} = m\,\hat{\mathbf{y}}$ in the $xy$ plane, with
$r^2 = x^2 + y^2$:

$$B_x = \frac{\mu_0 m}{4\pi}\frac{3xy}{r^5}
\qquad
B_y = \frac{\mu_0 m}{4\pi}\frac{3y^2 - r^2}{r^5}$$

Two checks worth remembering. On the axis ($x=0$, $y=d$) the field is
$+2\mu_0 m / 4\pi d^3$, and on the equator ($x=d$, $y=0$) it is
$-\mu_0 m / 4\pi d^3$. The field directly above a bar magnet is **twice as
strong** as the field the same distance out to the side, and points the
opposite way.

To an accelerator physicist, a "dipole magnet" means something else entirely:
a magnet whose bore field is **uniform**. We get to that in section 5. The two
usages are related only in that both have two poles.

In [ ]:
# Cell 02 - The field of a point magnetic dipole


def point_dipole_field(
    x: np.ndarray, y: np.ndarray, moment: float = 1.0
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute the field of a point magnetic dipole pointing along +y.

    Parameters
    ----------
    x, y : np.ndarray
        Coordinates in meters, measured from the dipole.
    moment : float
        Magnitude of the magnetic moment in A m^2.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        The (Bx, By) components of the field in tesla.
    """
    r = np.hypot(x, y)
    prefactor = mu_0 * moment / (4 * np.pi * r**5)
    return prefactor * 3 * x * y, prefactor * (3 * y**2 - r**2)


# Check the 2:1 ratio between the axis and the equator at the same distance
distance = 0.02  # 20 mm from the dipole
_, b_axis = point_dipole_field(np.array(0.0), np.array(distance))
_, b_equator = point_dipole_field(np.array(distance), np.array(0.0))

print(f"By on the axis    (0, {distance * 1e3:.0f} mm) = {b_axis * 1e3:+8.3f} mT")
print(f"By on the equator ({distance * 1e3:.0f} mm, 0) = {b_equator * 1e3:+8.3f} mT")
print(f"ratio = {b_axis / b_equator:+.3f}   (expected -2.000)")

In [ ]:
# Cell 03 - Plot 1: field lines around a bar magnet

span = 0.05  # plot out to 50 mm
plot_axis = np.linspace(-span, span, 400)
grid_x, grid_y = np.meshgrid(plot_axis, plot_axis)

field_x, field_y = point_dipole_field(grid_x, grid_y, moment=1.0)

# The point dipole formula diverges at the origin, so blank out the volume the
# bar magnet itself occupies. The formula is only valid outside the magnet.
MAGNET_HALF_WIDTH = 0.004
MAGNET_HALF_LENGTH = 0.008
inside_magnet = (np.abs(grid_x) <= MAGNET_HALF_WIDTH) & (
    np.abs(grid_y) <= MAGNET_HALF_LENGTH
)

field_magnitude = np.where(inside_magnet, np.nan, np.hypot(field_x, field_y))
field_x = np.where(inside_magnet, np.nan, field_x)
field_y = np.where(inside_magnet, np.nan, field_y)

fig, ax = plt.subplots(figsize=(7.5, 6.5))

shading = ax.contourf(
    grid_x * 1e3, grid_y * 1e3, np.log10(field_magnitude), levels=25, cmap="rainbow"
)
fig.colorbar(shading, ax=ax, shrink=0.8, label=r"$\log_{10}|B|$  (B in tesla)")

ax.streamplot(
    plot_axis * 1e3,
    plot_axis * 1e3,
    field_x,
    field_y,
    color="k",
    density=1.4,
    linewidth=0.8,
    arrowsize=0.9,
)

# Sketch the bar magnet that the point dipole is standing in for
ax.add_patch(
    plt.Rectangle(
        (-MAGNET_HALF_WIDTH * 1e3, -MAGNET_HALF_LENGTH * 1e3),
        2 * MAGNET_HALF_WIDTH * 1e3,
        2 * MAGNET_HALF_LENGTH * 1e3,
        facecolor="0.25",
        edgecolor="k",
    )
)
ax.text(0, 4.5, "N", color="w", ha="center", va="center", fontweight="bold")
ax.text(0, -4.5, "S", color="w", ha="center", va="center", fontweight="bold")

ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title("Field of a Point Magnetic Dipole (a Bar Magnet)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

The field lines loop from the north pole around to the south pole and never
start or stop anywhere - that is $\nabla \cdot \mathbf{B} = 0$ drawn as a
picture. Compare it with the capacitor plot from `parallel_plates.py`, where
the lines terminate on charges. There is no magnetic charge for them to
terminate on.

This is the field **outside** a magnet, and it falls off as $1/r^3$. It is a
poor way to steer a particle beam. What an accelerator needs is a field
sculpted *inside* a gap, and for that we go back to Laplace's equation.

---
## 3. The multipole expansion

An accelerator magnet is long compared with its bore, so near the middle
nothing depends on the beam direction $z$ and the problem is two dimensional.
We need the general solution of $\nabla^2 \psi = 0$ in a circular bore.

In polar coordinates,

$$\nabla^2 \psi
= \frac{\partial^2 \psi}{\partial r^2}
+ \frac{1}{r}\frac{\partial \psi}{\partial r}
+ \frac{1}{r^2}\frac{\partial^2 \psi}{\partial \theta^2} = 0$$

Separating variables with $\psi = R(r)\Theta(\theta)$ gives
$\Theta'' = -n^2\Theta$, so $\Theta$ is $\sin n\theta$ or $\cos n\theta$ with
$n$ an integer (the field must repeat after one turn). The radial equation is
then $r^2 R'' + rR' - n^2 R = 0$, an Euler equation whose solutions are
$r^{n}$ and $r^{-n}$. The bore includes $r = 0$, where $r^{-n}$ blows up, so
only the $r^{n}$ terms survive:

$$\psi(r,\theta) = -\sum_{n=1}^{\infty} \frac{C_n}{n}\, r^{n} \sin n\theta$$

(The $\cos$ terms describe *skew* magnets, the same magnets rotated; we take
the symmetry axes to line up with $x$ and $y$ and drop them.)

Each $n$ is one **multipole**, and the sign and the $1/n$ are conventional
bookkeeping that make the field come out clean. Since
$r^n \sin n\theta = \operatorname{Im}(z^n)$ for $z = x + iy$, the whole family
collapses into one complex statement:

$$\psi = -\operatorname{Im}\!\left(\sum_n \frac{C_n}{n} z^{n}\right)
\qquad\Longrightarrow\qquad
\boxed{\,B_y + i B_x = \sum_n C_n z^{\,n-1}\,}$$

That boxed line is the standard multipole expansion used to specify every
magnet in an accelerator. Working it out term by term:

| $n$ | name | $\psi$ | $B_x$ | $B_y$ |
| --- | --- | --- | --- | --- |
| 1 | dipole | $-B_0\,y$ | $0$ | $B_0$ |
| 2 | quadrupole | $-G\,xy$ | $G\,y$ | $G\,x$ |
| 3 | sextupole | $-\frac{S}{3}(3x^2y - y^3)$ | $2S\,xy$ | $S(x^2 - y^2)$ |

The dipole field is constant, the quadrupole field grows linearly from zero on
the axis, the sextupole quadratically, and so on. This is nothing more than a
Taylor expansion of the field about the beam axis, organized so that each term
is separately a solution of Laplace's equation.

In [ ]:
# Cell 04 - The multipole family, straight from the complex formula


def multipole_potential(
    x: np.ndarray, y: np.ndarray, order: int, strength: float = 1.0
) -> np.ndarray:
    """
    Evaluate the scalar potential of a single multipole.

    Parameters
    ----------
    x, y : np.ndarray
        Coordinates in meters.
    order : int
        Multipole order n. 1 is a dipole, 2 a quadrupole, 3 a sextupole.
    strength : float
        Coefficient C_n, in tesla per meter^(n-1).

    Returns
    -------
    np.ndarray
        The scalar potential psi in tesla-meters.
    """
    z = x + 1j * y
    return -(strength / order) * (z**order).imag


def multipole_field(
    x: np.ndarray, y: np.ndarray, order: int, strength: float = 1.0
) -> tuple[np.ndarray, np.ndarray]:
    """
    Evaluate the field of a single multipole using B_y + i B_x = C_n z^(n-1).

    Parameters
    ----------
    x, y : np.ndarray
        Coordinates in meters.
    order : int
        Multipole order n.
    strength : float
        Coefficient C_n.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        The (Bx, By) components of the field in tesla.
    """
    w = strength * (x + 1j * y) ** (order - 1)
    return w.imag, w.real


# Confirm the complex formula reproduces the hand written table above
xt, yt = 0.011, 0.007  # an arbitrary test point in the bore

hand_written = {
    "dipole (n=1)": (0.0, 1.0),
    "quadrupole (n=2)": (1.0 * yt, 1.0 * xt),
    "sextupole (n=3)": (2.0 * xt * yt, xt**2 - yt**2),
}

rows = []
for n, (name, (bx_hand, by_hand)) in enumerate(hand_written.items(), start=1):
    bx, by = multipole_field(xt, yt, order=n)
    rows.append(
        {
            "multipole": name,
            "Bx (formula)": bx,
            "Bx (by hand)": bx_hand,
            "By (formula)": by,
            "By (by hand)": by_hand,
        }
    )

pd.DataFrame(rows).set_index("multipole")

---
## 4. Pole faces are the level curves of $\psi$

Now the payoff. Section 1 established that an iron pole face is a surface of
constant $\psi$. Section 3 gives us $\psi$ for each multipole. Put those
together and the shape of the iron is no longer a matter of taste - it is
**dictated**:

> To build a pure $n$-pole magnet, machine the pole faces onto the level
> curves of $\psi = -\frac{C_n}{n} r^n \sin n\theta$.

For a **dipole** ($n = 1$), $\psi = -B_0 y$, so the level curves are the lines
$y = \text{constant}$: two **flat, parallel** pole faces. A magnet with flat
poles produces a uniform field, and now we know why.

For a **quadrupole** ($n = 2$), $\psi = -G\,xy$, so the level curves are
$xy = \text{constant}$: **rectangular hyperbolas**. If we want the pole tip a
distance $a$ (the bore radius) from the axis, the tip sits on the diagonal at
$x = y = a/\sqrt{2}$, so the four pole faces lie on

$$xy = \pm \frac{a^2}{2}
\qquad\text{at potentials}\qquad
\psi = \mp \frac{G a^2}{2}$$

The excitation follows from Ampere's law,
$\oint \mathbf{H}\cdot d\boldsymbol{\ell} = NI$. Taking a path from the axis
(where $\psi = 0$) out to a pole face and neglecting the drop through the
iron, the ampere-turns needed are $\Delta\psi/\mu_0$:

$$NI_{\text{dipole}} = \frac{B_0\, g}{\mu_0}
\qquad\qquad
NI_{\text{quadrupole}} = \frac{G a^2}{2\mu_0}$$

with $g$ the full gap (the dipole figure is for the whole gap, the quadrupole
figure is per pole). These are the two formulas a magnet engineer starts from,
and we will check both numerically.

### Setting up the numerical solve

The plan is exactly `parallel_plates.py`: fill a grid with zeros, pin the iron
regions to their fixed potentials, average with neighbors until converged.

One practical warning first, because it cost real time to find. **Build the
mesh so that the pole surfaces land exactly on grid lines.** If a pole face at
$y = 10\,\text{mm}$ falls half a cell off the mesh, the solver quietly models a
magnet with a slightly wider gap, and every field you extract is several
percent wrong with no error message anywhere. Here we lay the mesh out in
whole millimeters and keep all the geometry in millimeters, so the comparisons
that build the pole masks are exact integer arithmetic.

In [ ]:
# Cell 05 - The relaxation solver, and a check against a known answer


def make_grid(
    span_mm: float, step_mm: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """
    Build a square mesh laid out in whole millimeters.

    Parameters
    ----------
    span_mm : float
        Half width of the domain in millimeters.
    step_mm : float
        Cell size in millimeters. Must divide span_mm exactly.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, float]
        The 1D axis in mm, the 2D x and y meshes in mm, and the cell size in m.
    """
    axis_mm = np.arange(-span_mm, span_mm + step_mm / 2, step_mm)
    mesh_x, mesh_y = np.meshgrid(axis_mm, axis_mm)
    return axis_mm, mesh_x, mesh_y, step_mm / 1000.0


def solve_laplace(
    fixed_mask: np.ndarray, fixed_values: np.ndarray, iterations: int = 5000
) -> np.ndarray:
    """
    Solve Laplace's equation by repeated four neighbor averaging.

    Parameters
    ----------
    fixed_mask : np.ndarray
        Boolean array, True wherever the potential is held fixed (the iron).
    fixed_values : np.ndarray
        The potential to hold at those cells, in tesla-meters.
    iterations : int
        Number of relaxation sweeps.

    Returns
    -------
    np.ndarray
        The relaxed scalar potential psi on the grid.
    """
    psi = np.zeros_like(fixed_values)
    psi[fixed_mask] = fixed_values[fixed_mask]
    for _ in range(iterations):
        # Average every four neighbor cells in the grid
        psi = convolve(psi, LAPLACE_KERNEL, mode="constant")
        # Reapply the pole potentials
        psi[fixed_mask] = fixed_values[fixed_mask]
    return psi


def magnetic_field(psi: np.ndarray, cell_m: float) -> tuple[np.ndarray, np.ndarray]:
    """Differentiate the potential to get B = -grad(psi), in tesla."""
    dpsi_dy, dpsi_dx = np.gradient(psi, cell_m)
    return -dpsi_dx, -dpsi_dy


# Verification: pin the outer frame to an exactly known harmonic function,
# psi = -G x y, and confirm the solver recovers it everywhere inside.
check_axis, check_x, check_y, check_cell = make_grid(60.0, 2.0)
exact = multipole_potential(check_x / 1000, check_y / 1000, order=2, strength=13.2)

frame = np.zeros(exact.shape, dtype=bool)
frame[0, :] = frame[-1, :] = frame[:, 0] = frame[:, -1] = True

relaxed = solve_laplace(frame, exact, iterations=20000)
worst = np.abs(relaxed - exact).max()

print(f"grid: {exact.shape[0]} x {exact.shape[1]} cells of {check_cell * 1e3:.1f} mm")
print(f"peak potential          = {np.abs(exact).max():.6f} T m")
print(f"worst error vs analytic = {worst:.3e} T m")
print(f"relative error          = {worst / np.abs(exact).max():.2e}")

The solver reproduces a known harmonic function to machine precision, so any
error from here on is in the *geometry* we hand it, not in the relaxation.

---
## 5. The dipole magnet: bending the beam

An accelerator dipole is two flat iron poles facing each other across a gap,
wrapped in coils. From section 4 we expect a uniform vertical field between
them.

Let us design one: a **20 mm full gap**, poles **60 mm wide**, and a target
field of **1.0 T**. Since $\psi = -B_0 y$, the pole surfaces at
$y = \pm 10\,\text{mm}$ must be held at

$$\psi = \mp B_0 \times 10\,\text{mm} = \mp 0.010\ \text{T m}$$

The reason this magnet is useful is the Lorentz force. A particle of momentum
$p$ and charge $q$ in a uniform field $B$ moves on a circle of radius $\rho$
given by the **magnetic rigidity**

$$B\rho = \frac{p}{q}$$

so a uniform field bends the beam by a fixed angle regardless of where in the
gap the particle happens to be. That last clause is the whole point, and it is
only true where the field really is uniform. Real magnets are judged by the
width of their **good field region**, the span over which the field holds
constant to a stated tolerance, and we will measure ours.

In [ ]:
# Cell 06 - Solve Laplace's equation inside a dipole magnet

GAP_HALF_MM = 10.0  # half of the 20 mm gap
POLE_HALF_WIDTH_MM = 30.0  # poles are 60 mm across
POLE_DEPTH_MM = 20.0  # thickness of the iron block we model
DESIGN_FIELD = 1.0  # tesla, the field we are aiming for

axis_mm, mesh_x, mesh_y, cell_m = make_grid(80.0, 1.0)

# psi = -B0 * y, so the upper pole sits at -psi_pole and the lower at +psi_pole
psi_pole = DESIGN_FIELD * GAP_HALF_MM / 1000

upper_pole = (
    (np.abs(mesh_x) <= POLE_HALF_WIDTH_MM)
    & (mesh_y >= GAP_HALF_MM)
    & (mesh_y <= GAP_HALF_MM + POLE_DEPTH_MM)
)
lower_pole = (
    (np.abs(mesh_x) <= POLE_HALF_WIDTH_MM)
    & (mesh_y <= -GAP_HALF_MM)
    & (mesh_y >= -GAP_HALF_MM - POLE_DEPTH_MM)
)

dipole_iron = upper_pole | lower_pole
dipole_values = np.zeros_like(mesh_x)
dipole_values[upper_pole] = -psi_pole
dipole_values[lower_pole] = +psi_pole

dipole_psi = solve_laplace(dipole_iron, dipole_values, iterations=5000)
dipole_bx, dipole_by = magnetic_field(dipole_psi, cell_m)

center = len(axis_mm) // 2
midplane_by = dipole_by[center]


def flat_half_width(x_mm: np.ndarray, profile: np.ndarray, tolerance: float) -> float:
    """Return the half width in mm over which a profile stays within tolerance."""
    middle = len(x_mm) // 2
    inside = np.abs(profile - profile[middle]) / np.abs(profile[middle]) <= tolerance
    low = middle
    while low > 0 and inside[low - 1]:
        low -= 1
    high = middle
    while high < len(x_mm) - 1 and inside[high + 1]:
        high += 1
    return min(-x_mm[low], x_mm[high])


print(f"pole potential      = {psi_pole:+.4f} T m")
print(
    f"By at the center    = {dipole_by[center, center]:.5f} T  "
    f"(design {DESIGN_FIELD:.1f} T)"
)
print(f"Bx at the center    = {dipole_bx[center, center]:.2e} T  (expected 0)")
print(f"excitation required = {2 * psi_pole / mu_0:,.0f} ampere-turns")
print()
for tolerance in (1e-3, 1e-2):
    half = flat_half_width(axis_mm, midplane_by, tolerance)
    print(f"good field region at {tolerance:.1%} : |x| <= {half:.0f} mm")

In [ ]:
# Cell 07 - Plot 2: the dipole field, and how flat it really is

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left panel: potential contours, field lines, and the iron
ax = axes[0]
shading = ax.contourf(mesh_x, mesh_y, dipole_psi * 1e3, levels=24, cmap="rainbow")
fig.colorbar(shading, ax=ax, shrink=0.85, label=r"$\psi$  (mT m)")

# Hide the field inside the iron, where the scalar potential has no meaning.
# minlength has to be lowered or the short field lines that cross the gap
# straight from one pole to the other are discarded as too short to draw.
ax.streamplot(
    axis_mm,
    axis_mm,
    np.where(dipole_iron, np.nan, dipole_bx),
    np.where(dipole_iron, np.nan, dipole_by),
    color="k",
    density=1.4,
    linewidth=0.7,
    arrowsize=0.9,
    minlength=0.02,
)

ax.contourf(
    mesh_x, mesh_y, dipole_iron.astype(float), levels=[0.5, 1.5], colors=["0.3"]
)
# B runs from high psi to low psi, so it leaves the lower pole: that pole is north
ax.text(0, 20, "S", color="w", ha="center", va="center", fontweight="bold")
ax.text(0, -20, "N", color="w", ha="center", va="center", fontweight="bold")
ax.plot(0, 0, "wo", ms=6, mec="k")
ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title("Dipole Magnet: Scalar Potential and Field Lines")
ax.set_aspect("equal")

# Right panel: the field along the midplane, where the beam travels
ax = axes[1]
ax.plot(axis_mm, midplane_by, lw=2, label=r"relaxed $B_y(x)$")
ax.axhline(DESIGN_FIELD, color="k", ls="--", lw=1, label="design field")
for edge in (-POLE_HALF_WIDTH_MM, POLE_HALF_WIDTH_MM):
    ax.axvline(edge, color="crimson", ls=":", lw=1.2)
ax.text(
    POLE_HALF_WIDTH_MM, 0.55, " pole edge", color="crimson", rotation=90, va="center"
)

good = flat_half_width(axis_mm, midplane_by, 1e-3)
ax.axvspan(-good, good, color="tab:green", alpha=0.15, label="good field region (0.1%)")

ax.set_xlabel("x along the midplane (mm)")
ax.set_ylabel(r"$B_y$ (T)")
ax.set_title("Uniform in the Middle, Fringe Field at the Edges")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center")

fig.suptitle("Magnetostatic Field Inside a Dipole Bending Magnet")
plt.tight_layout()
plt.show()

The left panel shows the potential falling in even steps across the gap, which
is what a constant field looks like: evenly spaced equipotentials, field lines
straight and parallel. Near the pole edges the equipotentials bend outward,
the field lines bulge, and the field weakens - the **fringe field**.

The right panel quantifies it. The field is flat to 0.1% only over the middle
$\pm 12\,\text{mm}$ of a 60 mm wide pole, and has fallen off badly well before
the pole edge at 30 mm. This ratio is a rule of thumb in magnet design: the
usable aperture is far smaller than the iron, and buying good field means
buying pole width you cannot use. It is also why real magnets carry **shims**,
small raised steps at the pole edges, which push the field back up where it
would otherwise sag.

Note also that the center field came out at the design value to five digits.
The imposed pole potentials fix the gap field exactly, which is the scalar
potential picture earning its keep: $B = \Delta\psi / g$.

---
## 6. The quadrupole magnet

A quadrupole is the $n = 2$ term:

$$\psi = -G\,xy
\qquad
B_x = G\,y
\qquad
B_y = G\,x$$

$G = \partial B_y / \partial x$ is the **gradient**, in tesla per meter. Three
consequences follow immediately from those three lines:

1. **The field is exactly zero on the axis.** A quadrupole has no field
   strength of its own, only a gradient. A particle riding the axis is
   undeflected.
2. **The field grows linearly with distance from the axis**, so the further
   off axis a particle strays, the harder it is pushed back. That is the
   definition of a lens.
3. **The pole faces are hyperbolas** $xy = \pm a^2/2$, with the four pole tips
   on the diagonals.

For a proton of charge $q$ moving along $+z$ with speed $v$, the Lorentz force
$\mathbf{F} = q\mathbf{v}\times\mathbf{B}$ works out to

$$F_x = -qvG\,x
\qquad
F_y = +qvG\,y$$

Horizontally the force is restoring - the magnet **focuses**. Vertically it
pushes the particle further out - the magnet **defocuses**. This is not a
design flaw and cannot be engineered away: Laplace's equation forces
$\partial B_y/\partial x = -\partial B_x / \partial y$ in the bore, so the two
planes always carry equal and opposite focusing. Every quadrupole focuses one
plane and defocuses the other. Getting net focusing in both planes out of
alternating F and D quadrupoles is the subject of `strong_focusing.ipynb` in
Session 20.

We will build one with a **30 mm bore radius** and a design gradient of
**13.2 T/m**, the same magnet used in that notebook.

In [ ]:
# Cell 08 - Solve Laplace's equation inside a quadrupole magnet

BORE_RADIUS_MM = 30.0  # axis to pole tip
DESIGN_GRADIENT = 13.2  # T/m
YOKE_RADIUS_MM = 65.0  # where the poles are truncated and meet the yoke

axis_mm, mesh_x, mesh_y, cell_m = make_grid(80.0, 1.0)

# The pole faces are the hyperbolas x*y = +/- a^2 / 2, held at psi = -G x y
psi_pole = DESIGN_GRADIENT * (BORE_RADIUS_MM / 1000) ** 2 / 2
radius_mm = np.hypot(mesh_x, mesh_y)

# Poles in quadrants I and III sit on x*y = +a^2/2, so they are at -psi_pole
pole_positive = (mesh_x * mesh_y >= BORE_RADIUS_MM**2 / 2) & (
    radius_mm <= YOKE_RADIUS_MM
)
pole_negative = (mesh_x * mesh_y <= -(BORE_RADIUS_MM**2) / 2) & (
    radius_mm <= YOKE_RADIUS_MM
)

quad_iron = pole_positive | pole_negative
quad_values = np.zeros_like(mesh_x)
quad_values[pole_positive] = -psi_pole
quad_values[pole_negative] = +psi_pole

quad_psi = solve_laplace(quad_iron, quad_values, iterations=5000)
quad_bx, quad_by = magnetic_field(quad_psi, cell_m)

center = len(axis_mm) // 2
quad_midplane_by = quad_by[center]

# Fit a straight line to the midplane field over the inner 20 mm
fit_region = np.abs(axis_mm) <= 20.0
measured_gradient, intercept = np.polyfit(
    axis_mm[fit_region] / 1000, quad_midplane_by[fit_region], 1
)
straight_line = measured_gradient * axis_mm / 1000 + intercept
nonlinearity = np.abs(quad_midplane_by[fit_region] - straight_line[fit_region]).max()

print(f"pole potential        = {psi_pole:.5f} T m")
print(
    f"|B| on the axis       = "
    f"{np.hypot(quad_bx[center, center], quad_by[center, center]):.3e} T"
)
print(
    f"measured gradient     = {measured_gradient:.3f} T/m  (design "
    f"{DESIGN_GRADIENT:.1f} T/m, "
    f"{100 * (measured_gradient / DESIGN_GRADIENT - 1):+.2f}%)"
)
print(f"worst nonlinearity    = {nonlinearity:.2e} T over |x| <= 20 mm")
print(f"ideal pole tip field  = {DESIGN_GRADIENT * BORE_RADIUS_MM / 1000:.4f} T")
print(f"excitation required   = {psi_pole / mu_0:,.0f} ampere-turns per pole")

In [ ]:
# Cell 09 - Plot 3: the quadrupole field and its linear gradient

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left panel: the field map with the hyperbolic poles drawn in
ax = axes[0]
shading = ax.contourf(
    mesh_x,
    mesh_y,
    np.where(quad_iron, np.nan, np.hypot(quad_bx, quad_by)),
    levels=np.linspace(0, 0.5, 21),
    cmap="rainbow",
    extend="max",
)
fig.colorbar(shading, ax=ax, shrink=0.85, label="|B| (T)")

ax.streamplot(
    axis_mm,
    axis_mm,
    np.where(quad_iron, np.nan, quad_bx),
    np.where(quad_iron, np.nan, quad_by),
    color="k",
    density=1.3,
    linewidth=0.7,
    arrowsize=0.9,
    minlength=0.02,
)
ax.contourf(mesh_x, mesh_y, quad_iron.astype(float), levels=[0.5, 1.5], colors=["0.3"])

# Label the pole polarities: field leaves the high potential poles (north)
for x_label, y_label, tag in [
    (31, 31, "S"),
    (-31, -31, "S"),
    (31, -31, "N"),
    (-31, 31, "N"),
]:
    ax.text(
        x_label, y_label, tag, color="w", ha="center", va="center", fontweight="bold"
    )

# The force on a proton travelling into the page: Fx = -qvGx, Fy = +qvGy
for sign in (-1, 1):
    ax.arrow(sign * 20, 0, -sign * 8, 0, width=1.0, color="w", ec="k", zorder=5)
    ax.arrow(0, sign * 20, 0, sign * 8, width=1.0, color="w", ec="k", zorder=5)
ax.plot(0, 0, "wo", ms=6, mec="k")

ax.set_xlim(-38, 38)
ax.set_ylim(-38, 38)
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title("Quadrupole: Zero on Axis, Focusing in x, Defocusing in y")
ax.set_aspect("equal")

# Right panel: the midplane field against a straight line
ax = axes[1]
inside_bore = np.abs(axis_mm) <= BORE_RADIUS_MM
ax.plot(
    axis_mm[inside_bore], quad_midplane_by[inside_bore], lw=2, label=r"relaxed $B_y(x)$"
)
ax.plot(
    axis_mm[inside_bore],
    DESIGN_GRADIENT * axis_mm[inside_bore] / 1000,
    "k--",
    lw=1.2,
    label=rf"ideal $B_y = {DESIGN_GRADIENT}\,x$",
)
ax.axvspan(-20, 20, color="tab:green", alpha=0.15, label="fit region")
ax.axhline(0, color="0.5", lw=0.8)
ax.axvline(0, color="0.5", lw=0.8)
ax.set_xlabel("x along the midplane (mm)")
ax.set_ylabel(r"$B_y$ (T)")
ax.set_title("The Gradient Is Linear Through Zero")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

fig.suptitle("Magnetostatic Field Inside a Quadrupole Focusing Magnet")
plt.tight_layout()
plt.show()

The field map shows the two things that matter. The center is dark blue: the
field really does vanish on the axis, to seventeen decimal places in the
printout above. And the field climbs steadily out toward the four pole tips,
reaching $Ga = 13.2 \times 0.03 = 0.396\,\text{T}$ at the tip - modest compared
with the dipole's 1 T, because a quadrupole's job is the *slope*, not the
strength.

The white arrows are the Lorentz force on a proton travelling into the page:
inward horizontally, outward vertically.

The right panel shows the relaxed field tracking a straight line through the
origin, with a worst departure from straightness of about $2\times10^{-4}$ T
on a $0.26$ T swing. The measured gradient, though, came out about 2% below
the 13.2 T/m we designed for, and that deficit is worth explaining rather than
hiding, because it is purely numerical. A hyperbola drawn on a square mesh
becomes a staircase, and every step of that staircase sits slightly outside
the true curve, which makes the modeled bore a little larger than 30 mm and
the gradient correspondingly weaker. If that diagnosis is right, halving the
cell size should roughly halve the error.

In [ ]:
# Cell 10 - Mesh refinement: is the 2% deficit really a discretization error?

refinement = []
for step_mm in (2.0, 1.0, 0.5):
    ref_axis, ref_x, ref_y, ref_cell = make_grid(80.0, step_mm)
    ref_radius = np.hypot(ref_x, ref_y)

    ref_positive = (ref_x * ref_y >= BORE_RADIUS_MM**2 / 2) & (
        ref_radius <= YOKE_RADIUS_MM
    )
    ref_negative = (ref_x * ref_y <= -(BORE_RADIUS_MM**2) / 2) & (
        ref_radius <= YOKE_RADIUS_MM
    )
    ref_values = np.zeros_like(ref_x)
    ref_values[ref_positive] = -psi_pole
    ref_values[ref_negative] = +psi_pole

    # Relaxation spreads information one cell per sweep, so a mesh with half
    # the cell size needs about four times as many sweeps to converge
    sweeps = int(3000 * (1.0 / step_mm) ** 2)
    ref_psi = solve_laplace(ref_positive | ref_negative, ref_values, iterations=sweeps)
    _, ref_by = magnetic_field(ref_psi, ref_cell)

    ref_center = len(ref_axis) // 2
    ref_fit = np.abs(ref_axis) <= 20.0
    slope, _ = np.polyfit(ref_axis[ref_fit] / 1000, ref_by[ref_center][ref_fit], 1)

    refinement.append(
        {
            "cell size (mm)": step_mm,
            "grid": f"{len(ref_axis)} x {len(ref_axis)}",
            "sweeps": sweeps,
            "G (T/m)": round(slope, 3),
            "error (%)": round(100 * (slope / DESIGN_GRADIENT - 1), 2),
        }
    )

pd.DataFrame(refinement).set_index("cell size (mm)")

The error shrinks with every refinement, and between the two finest meshes it
halves when the cell size halves - the first order convergence that a
staircased curved boundary produces. The physics is right; it is the mesh that
is 2% off, and extrapolating the sequence to zero cell size lands on the
design gradient.

---
## 7. The AGS: putting both fields in one magnet

The Alternating Gradient Synchrotron at Brookhaven has a claim on this
material, because the strong focusing principle it is named for was worked out
there by Courant, Livingston, and Snyder in 1952. The machine first ran in
1960, accelerates protons around a ring roughly 807 m in circumference, and
reaches a momentum near 33 GeV/c.

Its main magnets, though, are neither dipoles nor quadrupoles. They are
**combined function** magnets: each of the roughly 240 of them both bends the
beam and focuses it, with the sign of the focusing alternating from one magnet
to the next. In our language that is simply the first two terms of the
multipole series kept together:

$$B_y(x) = B_0 + G\,x
\qquad
\psi = -\left(B_0\,y + G\,xy\right) = -y\,(B_0 + Gx)$$

Accelerator physicists usually quote the gradient through the dimensionless
**field index**

$$n = -\frac{\rho}{B_0}\frac{\partial B_y}{\partial x}$$

Weak focusing machines, everything built before 1952, needed $0 < n < 1$. The
alternating gradient idea removed that ceiling: $n$ may now be in the
hundreds, provided its sign alternates from magnet to magnet. That is where
the enormous increase in focusing strength comes from, and why the AGS vacuum
chamber is small enough to be affordable at 33 GeV where the earlier Cosmotron
needed thousands of tons of steel at 3 GeV. The numbers below are round
figures chosen to put the model at the right scale, not a magnet print.

The pole shape follows from $\psi$ as always. Setting $y\,(B_0 + Gx)$ constant
gives a hyperbola whose asymptote is the vertical line $x = -B_0/G$, so a
combined function magnet is nothing more than **a quadrupole with the beam
running off center**. The gap is tapered: narrower where the field must be
stronger.

Because that tapered pole is a curve rather than a flat face, we use a 0.5 mm
mesh here and snap the surface to the *nearest* grid line instead of the first
one beyond it. Section 6 showed what happens otherwise.

In [ ]:
# Cell 11 - AGS scale numbers and the combined function field

BRHO_PER_GEV = 3.3356  # T m of rigidity per GeV/c for a singly charged particle
AGS_MOMENTUM = 33.0  # GeV/c
AGS_BEND_RADIUS = 85.4  # m, radius of curvature inside the main magnets
AGS_FIELD_INDEX = 350.0  # representative of a strong focusing machine
AGS_GAP_HALF_MM = 10.0  # half gap on the beam axis

ags_rigidity = BRHO_PER_GEV * AGS_MOMENTUM
ags_field = ags_rigidity / AGS_BEND_RADIUS
ags_gradient = AGS_FIELD_INDEX * ags_field / AGS_BEND_RADIUS

print(f"magnetic rigidity B*rho = {ags_rigidity:8.2f} T m   at {AGS_MOMENTUM} GeV/c")
print(f"main magnet field B0    = {ags_field:8.3f} T     for rho = {AGS_BEND_RADIUS} m")
print(
    f"gradient G              = {ags_gradient:8.3f} T/m   for n = {AGS_FIELD_INDEX:.0f}"
)
print(f"equivalent quad center  = {-1e3 * ags_field / ags_gradient:8.1f} mm off axis")
print()

# The same relaxation solve, now with a tapered combined function pole
ags_axis_mm, ags_x, ags_y, ags_cell = make_grid(80.0, 0.5)

# psi = -y (B0 + G x), so the pole through (0, +/- gap_half) sits at this value
ags_psi_pole = ags_field * AGS_GAP_HALF_MM / 1000

# Solve psi = -psi_pole for y: the upper pole surface, in millimeters
pole_surface_mm = 1e3 * ags_psi_pole / (ags_field + ags_gradient * ags_x / 1000)

# Snap the curved surface to the nearest grid line rather than the next one out
half_cell_mm = 1e3 * ags_cell / 2
upper_iron = (ags_y >= pole_surface_mm - half_cell_mm) & (np.abs(ags_x) <= 60)
lower_iron = (ags_y <= -pole_surface_mm + half_cell_mm) & (np.abs(ags_x) <= 60)

ags_iron = upper_iron | lower_iron
ags_values = np.zeros_like(ags_x)
ags_values[upper_iron] = -ags_psi_pole
ags_values[lower_iron] = +ags_psi_pole

ags_psi = solve_laplace(ags_iron, ags_values, iterations=4000)
ags_bx, ags_by = magnetic_field(ags_psi, ags_cell)

ags_center = len(ags_axis_mm) // 2
ags_midplane = ags_by[ags_center]
in_aperture = np.abs(ags_axis_mm) <= 40.0
fitted_gradient, fitted_field = np.polyfit(
    ags_axis_mm[in_aperture] / 1000, ags_midplane[in_aperture], 1
)

print(f"gap on axis             = {2 * AGS_GAP_HALF_MM:.0f} mm")
print(
    f"relaxed B0 on the axis  = {fitted_field:8.3f} T     (asked for {ags_field:.3f})"
)
print(
    f"relaxed gradient        = {fitted_gradient:8.3f} T/m   "
    f"(asked for {ags_gradient:.3f})"
)

In [ ]:
# Cell 12 - Plot 4: a combined function magnet is a dipole plus a quadrupole

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left panel: the tapered gap and the field it produces
ax = axes[0]
shading = ax.contourf(
    ags_x,
    ags_y,
    np.where(ags_iron, np.nan, np.hypot(ags_bx, ags_by)),
    levels=np.linspace(1.0, 1.6, 25),
    cmap="rainbow",
    extend="both",
)
fig.colorbar(shading, ax=ax, shrink=0.55, label="|B| (T)")
ax.streamplot(
    ags_axis_mm,
    ags_axis_mm,
    np.where(ags_iron, np.nan, ags_bx),
    np.where(ags_iron, np.nan, ags_by),
    color="k",
    density=2.0,
    linewidth=0.7,
    arrowsize=0.9,
    minlength=0.01,
)
ax.contourf(ags_x, ags_y, ags_iron.astype(float), levels=[0.5, 1.5], colors=["0.3"])
ax.plot(0, 0, "wo", ms=6, mec="k")
ax.text(-34, 14, "gap widens,\nfield falls", color="w", fontsize=9, ha="center")
ax.text(34, 14, "gap narrows,\nfield rises", color="w", fontsize=9, ha="center")
ax.set_xlim(-55, 55)
ax.set_ylim(-21, 21)
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title("Combined Function Magnet: a Tapered Gap")
ax.set_aspect("equal")

# Right panel: the superposition, term by term
ax = axes[1]
x_m = ags_axis_mm[in_aperture] / 1000
ax.plot(
    ags_axis_mm[in_aperture],
    ags_midplane[in_aperture],
    lw=2.5,
    label=r"relaxed $B_y(x)$",
)
ax.plot(
    ags_axis_mm[in_aperture],
    np.full_like(x_m, ags_field),
    "--",
    lw=1.4,
    label=rf"dipole term $B_0 = {ags_field:.2f}$ T",
)
ax.plot(
    ags_axis_mm[in_aperture],
    ags_gradient * x_m,
    "--",
    lw=1.4,
    label=rf"quadrupole term $Gx$,  $G = {ags_gradient:.2f}$ T/m",
)
ax.plot(
    ags_axis_mm[in_aperture],
    ags_field + ags_gradient * x_m,
    "k:",
    lw=1.6,
    label=r"sum $B_0 + Gx$",
)
ax.axhline(0, color="0.5", lw=0.8)
ax.axvline(0, color="0.5", lw=0.8)
ax.set_ylim(-0.4, 2.15)
ax.set_xlabel("x along the midplane (mm)")
ax.set_ylabel(r"$B_y$ (T)")
ax.set_title("One Magnet, Two Multipoles")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", fontsize=9)

fig.suptitle("An AGS Style Combined Function Main Magnet")
plt.tight_layout()
plt.show()

The relaxed field reproduces $B_0 + Gx$ across the aperture: a bending field
with a linear gradient riding on top of it. Physically the beam sees a dipole,
and a particle displaced from the axis additionally sees a quadrupole.
Reversing the taper in the next magnet flips the sign of $G$ while leaving
$B_0$ alone, and that is the alternating gradient.

The tapered gap in the left panel is the visible signature of a combined
function magnet, and it is the same hyperbola we machined for the quadrupole,
just with its center pushed about 24 cm off to one side.

---
## 8. The multipole family

Everything in this notebook came from one series and one boundary condition.
Since $\psi = -\frac{C_n}{n}r^n\sin n\theta$, and iron sits on the level curves
of $\psi$, the pole geometry of any magnet follows from its multipole order
alone: an $n$-pole magnet has $2n$ poles spaced $180^\circ/n$ apart, and its
field grows as $r^{n-1}$.

Below, all three magnets are given the same 30 mm bore and the same 0.4 T at
the pole tip, so only the *shape* of the field differs. The heavy black curves
are the level curves of $\psi$ - the pole faces you would machine. Only the
bore is shaded, since that is the region where the multipole formula holds;
the dipole's bore is a single flat color because its field is the same
everywhere in it.

In [ ]:
# Cell 13 - Plot 5: the first three multipoles, side by side

APERTURE = 0.030  # 30 mm bore radius for all three magnets
TIP_FIELD = 0.4  # tesla at the pole tip, so the plots are comparable

family_axis = np.linspace(-0.055, 0.055, 400)
family_x, family_y = np.meshgrid(family_axis, family_axis)
family_radius = np.hypot(family_x, family_y)

fig, axes = plt.subplots(1, 3, figsize=(14, 5.2))

for ax, (order, name) in zip(
    axes, [(1, "Dipole"), (2, "Quadrupole"), (3, "Sextupole")], strict=True
):
    # Pick C_n so every magnet reaches the same field at the pole tip
    strength = TIP_FIELD / APERTURE ** (order - 1)

    bx, by = multipole_field(family_x, family_y, order, strength)
    psi = multipole_potential(family_x, family_y, order, strength)

    # The pole tips sit where |psi| is largest on the circle r = a
    tip_potential = strength * APERTURE**order / order

    # Shade only inside the bore. Outside it there is iron, not vacuum, so the
    # multipole formula does not apply there in a real magnet.
    shading = ax.contourf(
        family_x * 1e3,
        family_y * 1e3,
        np.where(family_radius <= APERTURE, np.hypot(bx, by), np.nan),
        levels=np.linspace(0, TIP_FIELD, 21),
        cmap="rainbow",
    )
    ax.streamplot(
        family_axis * 1e3,
        family_axis * 1e3,
        bx,
        by,
        color="k",
        density=1.0,
        linewidth=0.6,
        arrowsize=0.8,
    )
    # The iron: the level curves psi = +/- tip_potential, outside the bore
    ax.contour(
        family_x * 1e3,
        family_y * 1e3,
        np.where(family_radius > APERTURE * 0.9, psi, np.nan),
        levels=[-tip_potential, tip_potential],
        colors="k",
        linestyles="solid",
        linewidths=2.5,
    )
    ax.add_patch(
        plt.Circle((0, 0), APERTURE * 1e3, fill=False, color="w", ls="--", lw=1)
    )

    ax.set_title(f"{name}  (n = {order}),  " + rf"$|B| \propto r^{{{order - 1}}}$")
    ax.set_xlabel("x (mm)")
    ax.set_xlim(-55, 55)
    ax.set_ylim(-55, 55)
    ax.set_aspect("equal")

axes[0].set_ylabel("y (mm)")
fig.colorbar(shading, ax=axes, shrink=0.85, label="|B| (T)")
fig.suptitle(
    "The Multipole Family: Pole Faces Are the Level Curves of the Scalar Potential"
)
plt.show()

---
## Summary

| | Dipole | Quadrupole | Sextupole |
| --- | --- | --- | --- |
| order $n$ | 1 | 2 | 3 |
| $\psi$ | $-B_0 y$ | $-Gxy$ | $-\frac{S}{3}(3x^2y - y^3)$ |
| $B_y$ on the midplane | $B_0$ | $Gx$ | $Sx^2$ |
| pole shape | flat planes | hyperbolas $xy = \pm a^2/2$ | cubic curves |
| number of poles | 2 | 4 | 6 |
| ampere-turns | $B_0 g / \mu_0$ (full gap) | $G a^2 / 2\mu_0$ (per pole) | $S a^3 / 3\mu_0$ (per pole) |
| job in a ring | bend the beam | focus the beam | correct chromaticity |

The line of reasoning, start to finish:

1. In the bore there is no current, so $\nabla\times\mathbf{B} = 0$ and the
   field is the gradient of a scalar potential $\psi$.
2. $\nabla\cdot\mathbf{B} = 0$ then makes $\psi$ satisfy Laplace's equation -
   the same PDE, and the same relaxation solver, as the electrostatics in
   `parallel_plates.py`.
3. Iron pole faces are equipotentials of $\psi$, exactly as conductors are
   equipotentials of $V$.
4. Solving Laplace's equation in a circular bore gives the multipole series,
   whose $n$th term has a field growing as $r^{n-1}$.
5. Machining the iron onto the level curves of a chosen term builds a magnet
   that produces that term: flat poles for a uniform bending field, hyperbolic
   poles for a linear focusing gradient, and a tapered hyperbola for the
   combined function magnets of the AGS.

The numerical work confirmed all of it: 1.0 T in the gap we designed for, a
field exactly zero on the quadrupole axis, a gradient linear to a part in
$10^3$ across the useful aperture, and a residual 2% error that refined away
with the mesh rather than staying put.